# RNA Structure Analysis

This notebook demonstrates how to work with RNA secondary structures using `seq_tools`.

## RNA Folding

RNA sequences can fold into secondary structures. The `fold` function uses ViennaRNA to predict the minimum free energy (MFE) structure.


In [7]:
from seq_tools import sequence_to_dataframe, fold
import pandas as pd

# Create a simple hairpin sequence
rna_seq = "GGGGUUUUCCCC"
df = sequence_to_dataframe(rna_seq, name="hairpin")

# Fold the sequence
df_folded = fold(df)

print("Sequence folding results:")
print(df_folded[["name", "sequence", "structure", "mfe", "ens_defect"]])

Sequence folding results:
      name      sequence     structure  mfe  ens_defect
0  hairpin  GGGGUUUUCCCC  ((((....)))) -5.9    0.378602


### Understanding the Output

- **structure**: Dot-bracket notation where `(` and `)` represent paired bases, and `.` represents unpaired bases
- **mfe**: Minimum free energy in kcal/mol (more negative = more stable)
- **ens_defect**: Ensemble defect (0-1 scale, lower is better, represents structural diversity)


In [8]:
# Example: Visualize the structure
sequences = [
    "GGGGUUUUCCCC",  # Simple hairpin
    "GCGAAAGC",  # Another hairpin
    "AUGCAUGCAUGC",  # Less structured
]

results = []
for seq in sequences:
    df = sequence_to_dataframe(seq, name=seq)
    df = fold(df)
    results.append(df)

df_results = pd.concat(results, ignore_index=True)

print("Multiple sequences folded:")
for _, row in df_results.iterrows():
    print(f"\nSequence: {row['sequence']}")
    print(f"Structure: {row['structure']}")
    print(f"MFE: {row['mfe']:.2f} kcal/mol")
    print(f"Ensemble defect: {row['ens_defect']:.3f}")

Multiple sequences folded:

Sequence: GGGGUUUUCCCC
Structure: ((((....))))
MFE: -5.90 kcal/mol
Ensemble defect: 0.379

Sequence: GCGAAAGC
Structure: ((....))
MFE: -0.10 kcal/mol
Ensemble defect: 0.993

Sequence: AUGCAUGCAUGC
Structure: ............
MFE: 0.00 kcal/mol
Ensemble defect: 1.849


## SequenceStructure Class

The `SequenceStructure` class allows you to work with sequences and their structures together. This is useful for structure-based analysis and pattern matching.


In [9]:
from seq_tools import SequenceStructure

# Create a SequenceStructure object
seq = "GGGGUUUUCCCC"
struct = "((((....))))"
ss = SequenceStructure(seq, struct)

print(f"Sequence: {ss.sequence}")
print(f"Structure: {ss.structure}")
print(f"Length: {len(ss)}")

# Access properties
paired_positions = ss.get_paired_positions()
print(f"\nPaired positions: {paired_positions}")
print(f"Number of base pairs: {len(paired_positions)}")

# Get unpaired positions (all positions not in base pairs)
all_paired_indices = set()
for i, j in paired_positions:
    all_paired_indices.add(i)
    all_paired_indices.add(j)
unpaired_positions = [i for i in range(len(ss)) if i not in all_paired_indices]
print(f"Unpaired positions: {unpaired_positions}")
print(f"Number of unpaired bases: {len(unpaired_positions)}")

Sequence: GGGGUUUUCCCC
Structure: ((((....))))
Length: 12

Paired positions: [(3, 8), (2, 9), (1, 10), (0, 11)]
Number of base pairs: 4
Unpaired positions: [4, 5, 6, 7]
Number of unpaired bases: 4


## Structure-Based Pattern Matching

You can search for structural patterns within sequences using `find_seq_struct`. This is powerful for finding specific motifs or structural elements.


In [10]:
from seq_tools import find_seq_struct

# Target structure (a larger RNA)
target_seq = "GGGGUUUUCCCCAAAGGGGUUUUCCCC"
target_struct = "((((....))))...((((....))))"
target = SequenceStructure(target_seq, target_struct)

# Search for a pattern (hairpin structure)
pattern_seq = "GGGGUUUUCCCC"
pattern_struct = "((((....))))"
pattern = SequenceStructure(pattern_seq, pattern_struct)

# Find all matches (returns RNASegment objects)
segments = find_seq_struct(target, pattern)

print(f"Target sequence: {target_seq}")
print(f"Target structure: {target_struct}")
print(f"\nPattern sequence: {pattern_seq}")
print(f"Pattern structure: {pattern_struct}")
print(f"\nFound {len(segments)} match(es):")

for i, segment in enumerate(segments, 1):
    print(f"\nMatch {i}:")
    # RNASegment has strands with all indices
    strand_indices = segment.strands[
        0
    ]  # First (and only) strand for single-strand match
    start = min(strand_indices)
    end = max(strand_indices)
    print(f"  Positions: {start} to {end}")
    print(f"  All strand positions: {strand_indices}")
    print(f"  Matched sequence: {target_seq[start:end+1]}")

Target sequence: GGGGUUUUCCCCAAAGGGGUUUUCCCC
Target structure: ((((....))))...((((....))))

Pattern sequence: GGGGUUUUCCCC
Pattern structure: ((((....))))

Found 2 match(es):

Match 1:
  Positions: 15 to 27
  All strand positions: [15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27]
  Matched sequence: GGGGUUUUCCCC

Match 2:
  Positions: 0 to 12
  All strand positions: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12]
  Matched sequence: GGGGUUUUCCCCA


## Structure-Aware Extinction Coefficient

When you have structure information, the extinction coefficient calculation can account for hypochromicity effects from base pairing.


In [11]:
from seq_tools import get_extinction_coeff

# Same sequence, with and without structure
rna_seq = "GGGGUUUUCCCC"
structure = "((((....))))"

# Without structure
ec_no_struct = get_extinction_coeff(rna_seq, "RNA", double_stranded=False)
print(f"Extinction coefficient (no structure): {ec_no_struct:,} M⁻¹cm⁻¹")

# With structure (accounts for hypochromicity)
ec_with_struct = get_extinction_coeff(
    rna_seq, "RNA", double_stranded=False, structure=structure
)
print(f"Extinction coefficient (with structure): {ec_with_struct:,} M⁻¹cm⁻¹")
print(f"\nDifference: {ec_no_struct - ec_with_struct:,} M⁻¹cm⁻¹")
print("(Structured RNA has lower extinction coefficient due to base pairing)")

Extinction coefficient (no structure): 109,500 M⁻¹cm⁻¹
Extinction coefficient (with structure): 105,193 M⁻¹cm⁻¹

Difference: 4,307 M⁻¹cm⁻¹
(Structured RNA has lower extinction coefficient due to base pairing)


## Working with Connectivity Lists

You can examine the pairing relationships in a structure using connectivity lists.


In [12]:
from seq_tools.structure import ConnectivityList

# Create a connectivity list
seq = "GGGGUUUUCCCC"
struct = "((((....))))"
conn_list = ConnectivityList(seq, struct)

print(f"Sequence: {seq}")
print(f"Structure: {struct}")
print(f"\nConnectivity list: {conn_list.connections}")

# Check which positions are paired
print("\nPairing information:")
for i in range(len(seq)):
    if conn_list.is_nucleotide_paired(i):
        paired_idx = conn_list.get_paired_nucleotide(i)
        basepair = conn_list.get_basepair(i)
        print(
            f"Position {i} ({seq[i]}) pairs with position {paired_idx} ({seq[paired_idx]}) -> {basepair}"
        )
    else:
        print(f"Position {i} ({seq[i]}) is unpaired")

Sequence: GGGGUUUUCCCC
Structure: ((((....))))

Connectivity list: [11, 10, 9, 8, -1, -1, -1, -1, 3, 2, 1, 0]

Pairing information:
Position 0 (G) pairs with position 11 (C) -> GC
Position 1 (G) pairs with position 10 (C) -> GC
Position 2 (G) pairs with position 9 (C) -> GC
Position 3 (G) pairs with position 8 (C) -> GC
Position 4 (U) is unpaired
Position 5 (U) is unpaired
Position 6 (U) is unpaired
Position 7 (U) is unpaired
Position 8 (C) pairs with position 3 (G) -> CG
Position 9 (C) pairs with position 2 (G) -> CG
Position 10 (C) pairs with position 1 (G) -> CG
Position 11 (C) pairs with position 0 (G) -> CG


## Summary

In this notebook, we've covered:

- ✅ RNA folding with ViennaRNA
- ✅ Understanding dot-bracket notation
- ✅ Working with SequenceStructure objects
- ✅ Structure-based pattern matching
- ✅ Structure-aware extinction coefficient calculations
- ✅ Analyzing pairing relationships with connectivity lists

Next, we'll explore batch processing with DataFrames in **04_dataframe_operations.ipynb**.
